In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import os
np.random.seed(42)

In [ ]:
# q8.1

# Load mnist data
mnist_data = np.load((f"data/mnist-data-hw3.npz"))
fields = "test_data", "training_data", "training_labels"
test_data = mnist_data[fields[0]]
training_data = mnist_data[fields[1]]
training_labels = mnist_data[fields[2]]

# Contrast-normalize the images before using the pixel values
def data_normalize(data):
    '''
    Normalize the data values. 
    Here we choose to divide the data values of an image 
    by the l2-norm of its values.
    '''
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    norms[norms == 0] = 1
    normalized_data = data / norms
    return normalized_data

train_normalized = data_normalize(training_data)
data_reshaped = train_normalized.reshape(train_normalized.shape[0], -1)
train_labels_unique = np.unique(training_labels)
means = {}
covs = {}
for label in train_labels_unique:
    data_with_label = data_reshaped[training_labels == label]
    mean = np.mean(data_with_label, axis=0)
    # cov matrix = (1/n) * sum(x_i - mean)(x_i - mean)^T
    cov = np.cov(data_with_label, rowvar=False, bias=True)
    means[label] = mean
    covs[label] = cov

In [ ]:
# q8.2 Visualize the cov matrix for a particular class (digit)
# Here save the figure of the cov matrices for all the digits

digits = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
for digit in digits:
    cov_digit = covs[digit]
    plt.imshow(cov_digit, cmap='grey')
    plt.colorbar()
    plt.title(f"Covariance matrix of digit {digit}")
    plt.xlabel("Pixel index")
    plt.ylabel("Pixel index")
    folder = 'q8_figs'
    if not os.path.exists(folder):
        os.makedirs(folder)
    plt.savefig(os.path.join(folder, f"cov_matrix_digit_{digit}.png"))
    plt.show()

In [ ]:
# q8.3 cope with the train-validation set splitting

def train_val_split(data, labels, val_size):
    """
    Split data and labels into training and validation sets.
    
    Args:
        data (np.ndarray): sample data
        labels (np.ndarray): corresponding labels
        val_size (int): validation set size
    
    Returns:
        train_data, train_labels, val_data, val_labels
    """
    assert len(data) == len(labels), "Data and labels must have the same length"
    n_samples = len(data)
    
    # Shuffle indices
    indices = np.random.permutation(n_samples)
    data = data[indices]
    labels = labels[indices]
    
    assert 0 < val_size < n_samples, "Invalid validation size"
    
    val_data = data[:val_size]
    val_labels = labels[:val_size]
    train_data = data[val_size:]
    train_labels = labels[val_size:]
    
    return train_data, train_labels, val_data, val_labels

# Split the training data into train set and validation set
validation_size = 10000
train_data, train_labels, val_data, val_labels = \
    train_val_split(data_reshaped, training_labels, validation_size)

In [ ]:
# q8.3 two Classes: QDA and LDA

class LDA():
    def __init__(self):
        self.means = None
        self.cov = None
        self.labels = None
        self.prior_probs = None

    def fit(self, data, labels):
        n_samples, n_features = data.shape

        self.labels = np.unique(labels)
        self.means = {}
        self.prior_probs = {}
        # cov matrix = 1 / n 
        #       * \sum_{classes}{\sum_{yi = class}{(X_i - mean_class)(X_i - mean_class)^T}}
        self.cov = np.zeros((n_features, n_features))
        
        for label in self.labels:
            data_with_label = data[labels == label]
            # calculate the mean of each label
            mean = np.mean(data_with_label, axis=0)
            self.means[label] = mean
            # calculate the each-label-term of the LDA cov matrix
            diff = data_with_label - mean
            cov = np.dot(diff.T, diff)
            self.cov += cov
            # calculate the prior probability of each class
            prior_prob = data_with_label.shape[0] / n_samples
            self.prior_probs[label] = prior_prob
        self.cov = self.cov / n_samples

    def predict(self, data):
        if self.means is None or self.cov is None:
            raise ValueError("Model not fitted yet!")

        n_samples = data.shape[0]
        n_pred_labels = len(self.labels)

        log_probs = {}
        cov_lda = self.cov
        for label in self.labels:
            mean = self.means[label]
            prior_prob = self.prior_probs[label]
            log_pdf = scipy.stats.multivariate_normal.logpdf(data, 
                                                             mean=mean, cov=cov_lda,
                                                             allow_singular=True) \
                        + np.log(prior_prob)
            log_probs[label] = log_pdf

        all_log_probs = np.column_stack([log_probs[label] for label in self.labels])
        idx_prob_max = np.argmax(all_log_probs, axis=1)
        labels_list = list(self.labels)
        pred_labels = np.array([labels_list[idx] for idx in idx_prob_max])

        return pred_labels

In [ ]:
# q8.3 two Classes: QDA and LDA

class QDA():
    def __init__(self):
        self.means = None
        self.covs = None
        self.labels = None
        self.prior_probs = None

    def fit(self, data, labels):
        n_samples, n_features = data.shape

        self.labels = np.unique(labels)
        self.means = {}
        self.prior_probs = {}
        self.covs = {}
        
        for label in self.labels:
            data_with_label = data[labels == label]
            n_samples_with_label = data_with_label.shape[0]
            # calculate the mean of each label
            mean = np.mean(data_with_label, axis=0)
            self.means[label] = mean
            # calculate the cov matrix of each label
            diff = data_with_label - mean
            cov = np.dot(diff.T, diff) / n_samples_with_label
            # Add the Q7b trick to avoid the singular cov matrix
            # epsilon = 1e-8
            # cov += epsilon * np.eye(cov.shape[0])
            self.covs[label] = cov
            # calculate the prior probability of each class
            prior_prob = n_samples_with_label / n_samples
            self.prior_probs[label] = prior_prob

    def predict(self, data):
        if self.means is None or self.covs is None:
            raise ValueError("Model not fitted yet!")

        n_samples = data.shape[0]
        n_pred_labels = len(self.labels)

        log_probs = {}
        for label in self.labels:
            mean = self.means[label]
            cov = self.covs[label]
            prior_prob = self.prior_probs[label]
            log_pdf = scipy.stats.multivariate_normal.logpdf(data, 
                                                             mean=mean, cov=cov,
                                                             allow_singular=True) \
                        + np.log(prior_prob)
            log_probs[label] = log_pdf

        all_log_probs = np.column_stack([log_probs[label] for label in self.labels])
        idx_prob_max = np.argmax(all_log_probs, axis=1)
        labels_list = list(self.labels)
        pred_labels = np.array([labels_list[idx] for idx in idx_prob_max])

        return pred_labels

In [ ]:
# q8.3a

num_training_points = [100, 200, 500, 1000, 2000, 5000, 10000, 30000, 50000]
lda_errs = []
for num in num_training_points:
    lda = LDA()
    lda.fit(train_data[:num], train_labels[:num])
    pred_labels = lda.predict(val_data)
    n_correct_samples = np.sum(pred_labels == val_labels)
    n_val_samples = validation_size
    err = 1 - n_correct_samples / n_val_samples
    lda_errs.append(err)

plt.plot(num_training_points, lda_errs) 
plt.xlabel("Number of training points")
plt.ylabel("Error Rate")
plt.title("LDA Classification")
plt.ylim((0, 1))
folder = 'q8_figs'
plt.savefig(os.path.join(folder, "LDA_num_training_points.png"))
plt.show()

In [ ]:
# q8.3b

num_training_points = [100, 200, 500, 1000, 2000, 5000, 10000, 30000, 50000]
qda_errs = []
for num in num_training_points:
    qda = QDA()
    qda.fit(train_data[:num], train_labels[:num])
    pred_labels = qda.predict(val_data)
    n_correct_samples = np.sum(pred_labels == val_labels)
    n_val_samples = validation_size
    err = 1 - n_correct_samples / n_val_samples
    qda_errs.append(err)

plt.plot(num_training_points, qda_errs) 
plt.xlabel("Number of training points")
plt.ylabel("Error Rate")
plt.title("QDA Classification")
plt.ylim((0, 1))
folder = 'q8_figs'
plt.savefig(os.path.join(folder, "QDA_num_training_points.png"))
plt.show()

In [ ]:
# q8.3d LDA per digit

digits = np.unique(train_labels)
num_training_points = [100, 200, 500, 1000, 2000, 5000, 10000, 30000, 50000]
val_errors = {digit: [] for digit in digits}

for digit in digits:
    print(f"\nProcessing digit {digit}...")
    for num in num_training_points:
        lda = LDA()
        lda.fit(train_data[:num], train_labels[:num])
        pred_labels = lda.predict(val_data)
        n_correct_samples_digit = np.sum((val_labels == digit) 
                                         & (pred_labels == val_labels))
        n_val_samples_digit = np.sum(val_labels == digit)
        digit_err = 1 - n_correct_samples_digit / n_val_samples_digit
        val_errors[digit].append(digit_err)

for digit in digits:
    plt.plot(num_training_points, val_errors[digit], label=str(digit))

plt.xlabel("Number of training samples per digit")
plt.ylabel("Validation error rate")
plt.title("LDA Classification")
plt.legend(title="Digit")
plt.ylim((0, 1))
folder = 'q8_figs'
plt.savefig(os.path.join(folder, "LDA_per_digit_num_training_points.png"))
plt.show()

In [ ]:
# q8.3d QDA per digit

digits = np.unique(train_labels)
num_training_points = [100, 200, 500, 1000, 2000, 5000, 10000, 30000, 50000]
val_errors = {digit: [] for digit in digits}

for digit in digits:
    print(f"\nProcessing digit {digit}...")
    for num in num_training_points:
        qda = QDA()
        qda.fit(train_data[:num], train_labels[:num])
        pred_labels = qda.predict(val_data)
        n_correct_samples_digit = np.sum((val_labels == digit) 
                                         & (pred_labels == val_labels))
        n_val_samples_digit = np.sum(val_labels == digit)
        digit_err = 1 - n_correct_samples_digit / n_val_samples_digit
        val_errors[digit].append(digit_err)

for digit in digits:
    plt.plot(num_training_points, val_errors[digit], label=str(digit))

plt.xlabel("Number of training samples per digit")
plt.ylabel("Validation error rate")
plt.title("QDA Classification")
plt.legend(title="Digit")
plt.ylim((0, 1))
folder = 'q8_figs'
plt.savefig(os.path.join(folder, "QDA_per_digit_num_training_points.png"))
plt.show()

In [ ]:
# q8.4 Kaggle MNIST
from scripts.save_csv import results_to_csv

# I choose LDA with total 60000 points as training data.
lda = LDA()
lda.fit(data_reshaped, training_labels)
test_normalized = data_normalize(test_data)
test_normalized_reshaped = test_normalized.reshape(test_normalized.shape[0], -1)
pred_test_labels = lda.predict(test_normalized_reshaped)
results_to_csv(pred_test_labels)